In [1]:
from upf_conv import UPFv1
from pathlib import Path
from subprocess import run
from datetime import datetime


In [4]:
tmp = "Thu Jan  4 11:15:10 2024"

datetime.strptime(
    tmp,
    "%a %b %d %H:%M:%S %Y"
)

datetime.datetime(2024, 1, 4, 11, 15, 10)

In [2]:
test_upfv1 = Path("./recipes/Pt/ccECP/Pt.ccECP.AREP.upf")
test_upfv2 = Path("./recipes/Pt/ccECP/Pt.ccECP.AREP.UPF2")
test_upfv1 = UPFv1.from_upf_file(test_upfv1)

In [3]:
test_upfv1.fix_upfconv_output(test_upfv2, None)

<UPF version="2.0.1">
<PP_INFO>
</PP_INFO>
<!-- END OF HUMAN READABLE SECTION -->
<PP_HEADER generated="Generated by new atomic code, or converted to UPF format" author="" date="" comment="" element="Pt" pseudo_type="NC" relativistic="no" is_ultrasoft="false" is_paw="false" is_coulomb="false" has_so="false" has_wfc="false" has_gipaw="false" paw_as_gipaw="false" core_correction="false" with_metagga_info="false" functional="SLA  PW   PBE  PBE" z_valence="18.000000000000000" total_psenergy="0.0000000000000000" wfc_cutoff="0.0000000000000000" rho_cutoff="0.0000000000000000" l_max="2" l_max_rho="0" l_local="0" mesh_size="1188" number_of_wfc="3" number_of_proj="2"/>
<PP_MESH mesh="1188" dx="1.2999999999999998E-002" xmin="-6.3058678208497883" rmax="117.73347136384746" zmesh="78.000000000000000">
<PP_R>
</PP_R>
<PP_RAB>
</PP_RAB>
</PP_MESH>
<PP_LOCAL size="1188">
</PP_LOCAL>
<PP_NONLOCAL>
<PP_BETA.1 index="1" label="" angular_momentum="1" cutoff_radius_index="1188" cutoff_radius="0.00000000000

In [ ]:
def get_old_upf_info(file: Path) -> str:
    """Extract the information in ``<PP_INFO>`` from an old-style UPF file."""

    with open(file, "r") as old:
        old_text = old.readlines()

    old_pp_info = []
    reading_info = False
    for line in old_text:
        if reading_info:
            if "</PP_INFO" in line:
                break
            else:
                old_pp_info.append(line)
        elif "<PP_INFO" in line:
            reading_info = True
        
    return old_pp_info

In [ ]:
recipes = Path("./recipes")

elem_dirs = []
for dir in recipes.iterdir():
    elem_dirs.append(dir)

In [ ]:
old_upf_files = []
for elem in elem_dirs:
    for pp_dir in elem.iterdir():
        if "ccECP" in pp_dir.name:
            for file in pp_dir.iterdir():
                if file.suffix == ".upf":
                    old_upf_files.append(file)



In [ ]:
print(old_upf_files)
print(len(old_upf_files))

In [ ]:
for file in old_upf_files:
    command = f"/home/brock/Documents/lab_git/q-e/build/bin/upfconv.x -u {file.resolve()!s}"
    run(command, shell=True)

In [ ]:
new_upf_files = []
for elem in elem_dirs:
    for pp_dir in elem.iterdir():
        if "ccECP" in pp_dir.name:
            for file in pp_dir.iterdir():
                if file.suffix == ".UPF2":
                    new_upf_files.append(file)

print(new_upf_files)
print(len(new_upf_files))

In [ ]:
upf_pairs = {}
for old in old_upf_files:
    old_name = str(old)[:-4]
    for new in new_upf_files:
        new_name = str(new)[:-5]
        if old_name == new_name:
            upf_pairs[old] = new

print(upf_pairs)
print(len(upf_pairs))

In [ ]:
for old, new in upf_pairs.items():
    old_info = get_old_upf_info(old)

    new_info = "    ".join(old_info)

    with open(new, "r") as new_file:
        new_lines = new_file.readlines()

    for line in old_info:
        if line.strip().startswith("#Opium version"):
            opium_version = line.strip().split()[-1]
        if line.strip().startswith("#Execution Date"):
            psp_make_date = line.strip().split(":", 1)[-1].strip()

    new_temp = []
    skip_info = False
    for line in new_lines:
        if line.strip().startswith("<PP_HEADER"):
            line = line.replace(
                'generated="Generated by new atomic code, or converted to UPF format"',
                f'generated="Generated with OPIUM {opium_version}, converted to UPF 2.0.1 with upfconv.x"'
            )
            date = datetime.strftime(datetime.today(), "%d %b %Y")
            line = line.replace(
                'comment=""',
                f'comment="Updated to UPF 2.0.1 by B. A. Dyer on {date}"'
            )
            line = line.replace(
                'date=""',
                f'date="{psp_make_date}"'
            )
            new_temp.append(line)
        elif not skip_info:
            if "<PP_INFO>" in line:
                skip_info = True
            new_temp.append(line)
        elif "</PP_INFO>" in line:
            skip_info = False
            new_temp.append(line)

    new_temp.insert(2, new_info)
    print(new.resolve())
    with open(str(new)+".with_info", "w") as new_file:
        new_file.write("".join(new_temp))
    break